**Data Wrangling – Pandas Practice Sheet**

Name: ____________________

This notebook contains solutions for Questions 1–30. Upload all nine CSV files when prompted. Keep the file names unchanged.

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np

**Q1. Load students_master.csv and exam_marks.csv and display first five rows.**

Both files are loaded into DataFrames. The common column is used to connect records from the two files.

In [ ]:
students = pd.read_csv("students_master.csv")
exam = pd.read_csv("exam_marks.csv")

display(students.head())
display(exam.head())

common_columns = students.columns.intersection(exam.columns)
print("Common columns:", common_columns.tolist())

**Q2. Perform an inner merge on Student_ID.**

An inner join keeps only Student_ID values that occur in both DataFrames.

In [ ]:
inner_result = pd.merge(students, exam, on="Student_ID", how="inner")
display(inner_result)

**Q3. Perform a left merge.**

A left join keeps every student from the master file. Missing exam details appear as NaN.

In [ ]:
left_result = pd.merge(students, exam, on="Student_ID", how="left")
display(left_result)

**Q4. Perform a right merge.**

A right join keeps every record from the exam marks file, including IDs not found in the master file.

In [ ]:
right_result = pd.merge(students, exam, on="Student_ID", how="right")
display(right_result)

**Q5. Perform an outer merge and display all Student_ID values.**

An outer join keeps IDs from either file.

In [ ]:
outer_result = pd.merge(students, exam, on="Student_ID", how="outer")
display(outer_result)
print("All Student_ID values:", outer_result["Student_ID"].tolist())

**Q6. Concatenate section_CA.csv and section_CB.csv vertically while preserving original indexes.**

Vertical concatenation adds the rows of the second DataFrame below the first and preserves the original index labels.

In [ ]:
section_ca = pd.read_csv("section_CA.csv")
section_cb = pd.read_csv("section_CB.csv")

vertical_result = pd.concat([section_ca, section_cb], axis=0)
display(vertical_result)

**Q7. Concatenate vertically using ignore_index=True.**

ignore_index=True creates a fresh continuous index starting from zero.

In [ ]:
vertical_reset = pd.concat([section_ca, section_cb], axis=0, ignore_index=True)
display(vertical_reset)
print("New index:", vertical_reset.index)

**Q8. Concatenate section tables along axis=1.**

axis=1 places columns side by side and aligns rows by their index labels.

In [ ]:
horizontal_result = pd.concat([section_ca, section_cb], axis=1)
display(horizontal_result)
print("Column labels:", horizontal_result.columns.tolist())

**Q9. Create two Series and concatenate them along axis=0.**

Concatenating Series along axis=0 places the values one after another.

In [ ]:
attendance_series = pd.Series(section_ca["Attendance"])
student_id_series = pd.Series(section_cb["Student_ID"])

series_result = pd.concat([attendance_series, student_id_series], axis=0)
display(series_result)

**Q10. Display missing values in each Marks column.**

isna() returns True wherever a value is missing.

In [ ]:
marks_original = pd.read_csv("marks_original.csv")
marks_updated = pd.read_csv("marks_updated.csv")

print("Missing values in original Marks:")
display(marks_original["Marks"].isna())

print("Missing values in updated Marks:")
display(marks_updated["Marks"].isna())

**Q11. Set Student_ID as index and use combine_first().**

combine_first() fills missing values in the first DataFrame using matching values from the second DataFrame.

In [ ]:
original_indexed = marks_original.set_index("Student_ID")
updated_indexed = marks_updated.set_index("Student_ID")

combined_first = original_indexed.combine_first(updated_indexed)
display(combined_first)

**Q12. Use combine() with np.maximum.**

np.maximum compares the two values. If either value is missing, the result is generally missing.

In [ ]:
maximum_result = original_indexed.combine(
    updated_indexed,
    np.maximum
)
display(maximum_result)

**Q13. Use combine() with np.fmax.**

np.fmax selects the available non-missing value when only one of the two values is missing. If both are present, it selects the higher value.

In [ ]:
fmax_result = original_indexed.combine(
    updated_indexed,
    np.fmax
)
display(fmax_result)

**Q14. Use pivot() on marks_long.csv.**

pivot() converts subject values into separate columns. Each Student_ID and Subject combination must be unique.

In [ ]:
marks_long = pd.read_csv("marks_long.csv")

wide_marks = marks_long.pivot(
    index="Student_ID",
    columns="Subject",
    values="Marks"
)
display(wide_marks)

**Q15. Use stack() to return subject columns to a hierarchical row index.**

stack() moves the column labels into the row index, creating a hierarchical index.

In [ ]:
stacked_marks = wide_marks.stack()
display(stacked_marks)

**Q16. Inner merge semester_enrollments.csv with students_master.csv.**

A student may appear in multiple rows because the enrollment file can contain more than one semester or course record for the same Student_ID.

In [ ]:
enrollments = pd.read_csv("semester_enrollments.csv")

enrollment_inner = pd.merge(
    enrollments,
    students,
    on="Student_ID",
    how="inner"
)
display(enrollment_inner)

**Q17. Left merge and identify unmatched enrollment records.**

Rows marked left_only are enrollment records without a matching student in the master file.

In [ ]:
enrollment_left = pd.merge(
    enrollments,
    students,
    on="Student_ID",
    how="left",
    indicator=True
)
display(enrollment_left)

unmatched_enrollments = enrollment_left[enrollment_left["_merge"] == "left_only"]
display(unmatched_enrollments)

**Q18. Compare row counts for all four join types.**

Inner keeps matching IDs, left keeps all master IDs, right keeps all exam IDs, and outer keeps IDs from both files.

In [ ]:
join_results = {
    "Inner": pd.merge(students, exam, on="Student_ID", how="inner"),
    "Left": pd.merge(students, exam, on="Student_ID", how="left"),
    "Right": pd.merge(students, exam, on="Student_ID", how="right"),
    "Outer": pd.merge(students, exam, on="Student_ID", how="outer")
}

for join_name, result in join_results.items():
    print(join_name, "join:", len(result), "rows")

**Q19. Concatenate sections vertically using the default join behavior.**

The default outer column behavior keeps columns from both files. A column absent from one file receives missing values for those rows.

In [ ]:
default_concat = pd.concat([section_ca, section_cb], axis=0)
display(default_concat)
print("Missing values by column:")
display(default_concat.isna().sum())

**Q20. Concatenate sections using join='inner' and ignore_index=True.**

join='inner' keeps only columns present in both DataFrames.

In [ ]:
inner_concat = pd.concat(
    [section_ca, section_cb],
    axis=0,
    join="inner",
    ignore_index=True
)
display(inner_concat)
print("Remaining columns:", inner_concat.columns.tolist())

**Q21. Compare combine_first() before and after setting Student_ID as index.**

Without setting Student_ID as the index, Pandas aligns rows by their existing index labels. Setting Student_ID as the index aligns marks by student, which is the appropriate method.

In [ ]:
direct_result = marks_original.combine_first(marks_updated)
print("Without setting Student_ID as index:")
display(direct_result)

indexed_result = (
    marks_original.set_index("Student_ID")
    .combine_first(marks_updated.set_index("Student_ID"))
)
print("After setting Student_ID as index:")
display(indexed_result)

**Q22. Use np.fmax and display marks in ascending Student_ID order.**

np.fmax selects the higher available mark for each student and sort_index() orders the Student_ID values.

In [ ]:
higher_marks = original_indexed.combine(updated_indexed, np.fmax)
higher_marks = higher_marks.sort_index()
display(higher_marks)

**Q23. Use unstack() to reconstruct the wide marks table.**

unstack() moves the innermost row-index level back into columns.

In [ ]:
reconstructed_wide = stacked_marks.unstack()
display(reconstructed_wide)

print("Does it match the pivot table?")
print(reconstructed_wide.equals(wide_marks))

**Q24. Attempt pivot() on repeat_tests.csv and explain the issue.**

pivot() cannot reshape duplicate Student_ID and Subject pairs because it requires one value for each pair.

In [ ]:
repeat_tests = pd.read_csv("repeat_tests.csv")

try:
    repeat_wide = repeat_tests.pivot(
        index="Student_ID",
        columns="Subject",
        values="Marks"
    )
    display(repeat_wide)
except ValueError as error:
    print(error)

**Q25. Use pivot_table() to calculate mean Marks.**

pivot_table() handles repeated pairs by aggregating their values. Here, it calculates the mean.

In [ ]:
repeat_mean = repeat_tests.pivot_table(
    index="Student_ID",
    columns="Subject",
    values="Marks",
    aggfunc="mean"
)
display(repeat_mean)

**Q26. Build a consolidated roster and merge it with exam marks.**

The left merge retains every student in the consolidated roster. Students with missing exam marks have NaN in one or both marks columns.

In [ ]:
class_roster = pd.concat(
    [section_ca, section_cb],
    ignore_index=True
)

roster_with_marks = pd.merge(
    class_roster,
    exam,
    on="Student_ID",
    how="left"
)

display(roster_with_marks[
    ["Student_ID", "Student_Name", "Section", "Python_Marks", "DBMS_Marks"]
])

missing_exam_marks = roster_with_marks[
    roster_with_marks[["Python_Marks", "DBMS_Marks"]].isna().any(axis=1)
]
print("Students with missing exam marks:")
display(missing_exam_marks)

**Q27. Combine marks by Student_ID and merge with the master list.**

combine_first() keeps the original mark when it is available and uses the updated mark when the original mark is missing. The displayed rows show the source of the final values for IDs 101 and 102.

In [ ]:
marks_by_student = (
    marks_original.set_index("Student_ID")[["Marks"]]
    .combine_first(
        marks_updated.set_index("Student_ID")[["Marks"]]
    )
)

final_student_marks = students.merge(
    marks_by_student.reset_index(),
    on="Student_ID",
    how="left"
)
display(final_student_marks)

print("Final value for Student_ID 101:")
display(final_student_marks[final_student_marks["Student_ID"] == 101])

print("Final value for Student_ID 102:")
display(final_student_marks[final_student_marks["Student_ID"] == 102])

**Q28. Compare combine_first() and np.fmax side by side.**

combine_first() prefers the original value whenever it is present. np.fmax() chooses the higher value when both values are available. The displayed rows identify where the results differ.

In [ ]:
original_series = marks_original.set_index("Student_ID")["Marks"]
updated_series = marks_updated.set_index("Student_ID")["Marks"]

comparison = pd.concat(
    [
        original_series.combine_first(updated_series).rename("combine_first"),
        original_series.combine(updated_series, np.fmax).rename("np_fmax")
    ],
    axis=1
).sort_index()

display(comparison)

different_ids = comparison[
    comparison["combine_first"] != comparison["np_fmax"]
]
display(different_ids)

**Q29. Calculate mean and maximum marks, reshape, and restore the pivot table.**

The pivot table uses two aggregation functions. stack() and unstack() move index and column levels while preserving the summarized values.

In [ ]:
repeat_summary = repeat_tests.pivot_table(
    index="Student_ID",
    columns="Subject",
    values="Marks",
    aggfunc=["mean", "max"]
)
display(repeat_summary)

reshaped_summary = repeat_summary.stack(level=0)
display(reshaped_summary)

restored_summary = reshaped_summary.unstack(level=-1)
display(restored_summary)

**Q30. Merge students with marks_long.csv, pivot, and reset_index().**

This pivot succeeds when each Student_ID and Subject combination is unique in marks_long.csv. The repeat_tests.csv file contains repeated combinations, so pivot() cannot reshape it without aggregation.

In [ ]:
student_marks_long = pd.merge(
    students,
    marks_long,
    on="Student_ID",
    how="inner"
)

final_wide_marks = student_marks_long.pivot(
    index=["Student_ID", "Student_Name"],
    columns="Subject",
    values="Marks"
).reset_index()

final_wide_marks.columns.name = None
display(final_wide_marks)